In [ ]:
import zipfile
with zipfile.ZipFile('trt_yolv4-tiny-master.zip', 'r') as z:
    z.extractall('.')

In [ ]:
!ls trt_yolv4-tiny-master/

In [ ]:
import gc
gc.collect()

In [1]:
import os
import time

os.chdir('trt_yolv4-tiny-master')

import pycuda.autoinit
from utils.yolo_classes import get_cls_dict
from utils.display import open_window, set_display, show_fps
from utils.visualization import BBoxVisualization
from utils.yolo import TRT_YOLO

trt_yolo = TRT_YOLO("yolov4-tiny-416", (416, 416), 4)

In [2]:
import cv2
import ipywidgets.widgets as widgets
from IPython.display import display

# 1. 讀取影像
img = cv2.imread('1.jpg')

# 2. 進行模型辨識
boxes, confs, clss = trt_yolo.detect(img, 0.3)

# 3. 跑迴圈將辨識結果畫在影像上
for i in range(len(clss)):
    cv2.rectangle(img, (boxes[i][0], boxes[i][1]), (boxes[i][2], boxes[i][3]), (255, 0, 0), 2)

# 將 OpenCV 的影像編碼成 JPEG 格式
_, encoded_image = cv2.imencode('.jpg', img)

# 建立一個 Image 元件並顯示
image_widget = widgets.Image(value=encoded_image.tobytes(), format='jpg', width=400)
display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [3]:
import traitlets
from jetbot import Camera, bgr8_to_jpeg
from jetbot import Robot

# 只有使用 USBCamera 的需要這幾行
# from jetcam.usb_camera import USBCamera
# camera = USBCamera(capture_device=0)
# camera.running = True

# 一般 camera (CSI 介面) 的用這一行
camera = Camera.instance(width=416, height=416)

image = widgets.Image(format='jpeg', width=224, height=224)
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

robot = Robot()
road_close = 0

In [4]:
import numpy as np

def getROI(img, vertices):
    mask = np.zeros_like(img)
    cv2.fillPoly(mask, vertices, (255,))
    return cv2.bitwise_and(img, mask)

def make_points(image, avg):
    slope, y_int = avg
    if abs(slope) < 0.01:
        slope = 0.01
    y1 = image.shape[0]        # 240
    y2 = int(y1 * 0.5)         # 120
    x1 = int((y1 - y_int) / slope)
    x2 = int((y2 - y_int) / slope)
    x1 = max(0, min(x1, 320))
    x2 = max(0, min(x2, 320))
    return np.array([x1, y1, x2, y2])


In [15]:
def average(image, lines):
    if lines is None:
        return None
    left, right = [], []
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        if x2 == x1:
            continue
        params = np.polyfit((x1, x2), (y1, y2), 1)
        slope, y_int = params[0], params[1]
        if abs(slope) < 0.5 or abs(slope) > 5:
            continue
        if slope < 0:
            left.append((slope, y_int))
        else:
            right.append((slope, y_int))
    if not left and not right:
        return None
    left_avg  = np.average(left,  axis=0) if left  else None
    right_avg = np.average(right, axis=0) if right else None
    if left_avg is None:
        virtual_mid = make_points(image, right_avg)
        virtual_mid[2] -= 30 # 往左偏移 10 像素
        return np.array([virtual_mid])
    if right_avg is None:
        virtual_mid = make_points(image, left_avg)
        virtual_mid[2] += 30 # 往右偏移 10 像素
        return np.array([virtual_mid])
    return np.array([make_points(image, left_avg),
                     make_points(image, right_avg)])


In [16]:
def getAverageLines(origin_road):
    # 縮小到 320x240 省運算
    origin_road = cv2.resize(origin_road, (320, 240))
    road_gray = cv2.cvtColor(origin_road, cv2.COLOR_BGR2GRAY)
    road_gray = cv2.GaussianBlur(road_gray, (3, 3), 0)
    edges = cv2.Canny(road_gray, 200, 200, apertureSize=3)
    # ROI 沿跑道白線校準（實際拍照確認）
    ROI_vertices = [(0,240),(0,160),(80,60),(120,40),(200,40),(240,60),(320,160),(320,240)]
    
    edges = getROI(edges, np.array([ROI_vertices], np.int32))
    linesp = cv2.HoughLinesP(edges, rho=1, theta=np.pi / 180,
                             threshold=30, minLineLength=15, maxLineGap=5)
    return average(origin_road, linesp)


In [24]:
slow_mode = 0
prev_error = 0

def modifySpeed(origin_road):
    global speed_left, speed_right, speed, slow_mode, prev_error
    averaged_lines = getAverageLines(origin_road)
    
    if averaged_lines is not None:
        mid_x = int(np.average(averaged_lines, axis=0)[2])
        center = 160
        error = mid_x - center
        
        # --- 新增：PD 控制邏輯 ---
        # P (比例)：對應當前位置誤差
        # D (微分)：對應誤差變化率（這是防甩尾的關鍵！）
        d_error = error - prev_error 
        
        # 調整這些係數，P 決定靈敏度，D 決定阻尼感
        # 將 P 調小一點以防暴衝，D 可以抵銷過快的轉向
        correction = (error * 0.0004) + (d_error * 0.0002)
        
        prev_error = error # 更新上一幀誤差
        
        # --- 關鍵：速度基準歸零 ---
        # 不要使用 (speed_left + correction)，改用基於原始速度 speed 的基礎調整
        # 這樣每一幀都是基於「空狀態」重新計算，不會有積累誤差
        factor = 0.7 if slow_mode else 1.0
        
        speed_left  = max(0.1, (speed + correction) * factor)
        speed_right = max(0.1, (speed - correction) * factor)

        robot.set_motors(speed_left, speed_right)
    else:
        # 保持你的緊急停車機制
        robot.stop()

In [25]:
import time
stop_ignore    = 0
railway_ignore = 0

def getNearest(signs):
    if not len(signs):
        return [0, 4]
    t = time.time()
    cls = signs[0][1]
    # class 0 封路：永久，不需要 ignore
    # class 1 慢速：不需要 ignore
    # class 2 鐵路：有 railway_ignore
    # class 3 停止：有 stop_ignore
    if cls == 0 \
        or cls == 1 \
        or (cls == 2 and t > railway_ignore) \
        or (cls == 3 and t > stop_ignore):
        return signs[0]
    return getNearest(signs[1:]) if len(signs) >= 2 else [0, 4]


In [26]:
road_close = 0
slow_mode  = 0
sign_hold  = [0, 4]
count      = 0
is_processing = 0

def update(change):
    # print("update", time.time())
    
    global robot, count, speed_left, speed_right
    global stop_ignore, railway_ignore
    global road_close, slow_mode, sign_hold, is_processing
    ALERT_WIDTH = 30
    
    if (is_processing):
        return 
    
    is_processing = 1
    
    try:
        img = change['new']
        # print("frame id", id(img), time.time())

        if road_close:
            robot.stop()
            return

        modifySpeed(img)

        if averaged_lines is None:
            robot.stop()
            return
        
        count += 1
        if count < 10:
            return
        count = 0

        # detect 同步跑，while(1) 不怕卡
        start = time.time()
        boxes, confs, clss = trt_yolo.detect(img, 0.3)
        # print("detect cost =", time.time() - start)

        signs = []
        for box, cls in zip(boxes, clss):
            signs.append([box[2] - box[0], cls])
        signs.sort(reverse=True, key=lambda x: x[0])
        sign = getNearest(signs)

        t = time.time()
        # print('sign:', sign)  # debug 確認有在跑

        if sign[1] == 0 and sign[0] > ALERT_WIDTH:
            print('road close')
            print("detect cost =", time.time() - start)
            road_close = 1
            robot.stop()
        elif sign[1] == 1 and sign[0] > ALERT_WIDTH:
            print('slow')
            print("detect cost =", time.time() - start)
            slow_mode = 1
        elif sign[1] == 2 and sign[0] > ALERT_WIDTH and t > railway_ignore:
            print('railway')
            print("detect cost =", time.time() - start)
            slow_mode = 0
            robot.stop()
            railway_ignore = t + 10
            time.sleep(5)
        elif sign[1] == 3 and sign[0] > ALERT_WIDTH and t > stop_ignore:
            print('stop')
            print("detect cost =", time.time() - start)
            slow_mode = 0
            robot.stop()
            stop_ignore = t + 10
            time.sleep(3)
        else:
            slow_mode = 0
            if road_close == 0:
                robot.set_motors(speed_left, speed_right)
    finally:
        is_processing = 0

In [30]:
import cv2
import ipywidgets.widgets as widgets
from IPython.display import display
import numpy as np

img = camera.value.copy()

# 偵測路牌
boxes, confs, clss = trt_yolo.detect(img, 0.3)

# 取得車道平均線
averaged_lines = getAverageLines(img)
if averaged_lines is not None:
    for x1, y1, x2, y2 in averaged_lines:
        # 座標從 320x240 縮放到 416x416
        cv2.line(img,
                 (int(x1 / 320 * 416), int(y1 / 240 * 416)),
                 (int(x2 / 320 * 416), int(y2 / 240 * 416)),
                 (0, 0, 255), 5)
    mid_x = int(np.average(averaged_lines, axis=0)[2])
    mid_scaled = int(mid_x / 320 * 416)
    cv2.line(img, (mid_scaled, 200), (mid_scaled, 215), (0, 255, 0), 3)
    print('mid: ', mid_x)

# 畫出路牌辨識框
for i in range(len(clss)):
    cv2.rectangle(img,
                  (boxes[i][0], boxes[i][1]),
                  (boxes[i][2], boxes[i][3]),
                  (0, 255, 0), 2)
    cv2.putText(img, str(int(clss[i])),
                (boxes[i][0], boxes[i][1] - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

_, encoded_image = cv2.imencode('.jpg', img)
image_widget = widgets.Image(value=encoded_image.tobytes(), format='jpg', width=416, height=416)
display(image_widget)


mid:  167


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [28]:
speed = speed_left = 0.185
speed_right = 0.175
# robot.set_motors(0.41, 0.37)
# time.sleep(0.3)
robot.set_motors(speed_left, speed_right)

while(1):
    update({'new': camera.value})


stop
detect cost = 0.04147481918334961
railway
detect cost = 0.04439282417297363
slow
detect cost = 0.04676651954650879
road close
detect cost = 0.04392123222351074


KeyboardInterrupt: 

In [29]:
speed = speed_left = speed_right = 0
slow_mode  = 0
road_close = 0
robot.stop()


In [ ]:
camera_link.unlink()

In [ ]:
camera.stop()

In [59]:
# Debug 1
img = cv2.resize(camera.value, (320, 240))
pts = np.array([(0,240),(0,160),(80,60),(120,40),(200,40),(240,60),(320,160),(320,240)], np.int32)
cv2.polylines(img, [pts], True, (0,255,0), 2)
_, enc = cv2.imencode('.jpg', img)
display(widgets.Image(value=enc.tobytes(), format='jpg', width=400))

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [57]:
# Debug 2
img = cv2.resize(camera.value, (320, 240))
road_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
road_gray = cv2.GaussianBlur(road_gray, (5, 5), 0)
edges = cv2.Canny(road_gray, 200, 200, apertureSize=3)


# 套上 ROI
ROI_vertices = [(0,240),(0,160),(80,60),(120,40),(200,40),(240,60),(320,160),(320,240)]
edges_roi = edges.copy()
mask = np.zeros_like(edges)
cv2.fillPoly(mask, np.array([ROI_vertices], np.int32), 255)
edges_roi = cv2.bitwise_and(edges, mask)

_, enc = cv2.imencode('.jpg', edges_roi)
display(widgets.Image(value=enc.tobytes(), format='jpg', width=400))

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [ ]:
# Debug 3
robot.set_motors(0.18, 0.17)

In [ ]:
# Debug 4
import traitlets
from jetbot import Camera, bgr8_to_jpeg
import ipywidgets.widgets as widgets
from IPython.display import display

# 1. 每次執行前，先嘗試清理可能殘留的舊相機物件
if 'camera' in locals():
    camera.stop()

try:
    # 2. 重新實例化相機
    camera = Camera.instance(width=416, height=416)
    image = widgets.Image(format='jpeg', width=224, height=224)
    camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)
    display(image)
    
    print("相機啟動成功！請在測試完畢後，務必執行下一個 Cell 來關閉相機。")
    
except Exception as e:
    print(f"相機啟動失敗: {e}")
    print("請前往 SSH 終端機執行: sudo systemctl restart nvargus-daemon")

In [ ]:
# Debug 5
# 測試完畢後，絕對要點擊執行這個 Cell
if 'camera_link' in locals():
    camera_link.unlink()
if 'camera' in locals():
    camera.stop()
    print("相機資源已安全釋放。")

In [ ]:
# Debug 6
import time

start = time.time()

boxes, confs, clss = trt_yolo.detect(img, 0.3)

print(time.time()-start)